# A1: Uber Fare Prediction (Linear Regression vs Random Forest)

Dataset: `uber.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from math import radians, sin, cos, sqrt, asin

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
df = pd.read_csv("uber.csv")

df.drop(['Unnamed: 0','key'], axis=1, inplace=True)
df.dropna(inplace=True)

df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.day
df['month'] = df['pickup_datetime'].dt.month

df.drop('pickup_datetime', axis=1, inplace=True)

df.head()

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=df['fare_amount'])
plt.show()

Q1 = df['fare_amount'].quantile(0.25)
Q3 = df['fare_amount'].quantile(0.75)
IQR = Q3 - Q1

df = df[
    (df['fare_amount'] >= Q1 - 1.5*IQR) &
    (df['fare_amount'] <= Q3 + 1.5*IQR)
]

In [ ]:
def haversine(lat1, lon1, lat2, lon2):

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat/2)**2 + \
        cos(lat1)*cos(lat2)*sin(dlon/2)**2

    c = 2 * asin(sqrt(a))

    return 6371 * c


df['distance_km'] = df.apply(
    lambda x: haversine(
        x['pickup_latitude'],
        x['pickup_longitude'],
        x['dropoff_latitude'],
        x['dropoff_longitude']
    ),
    axis=1
)

df = df[df['distance_km'] > 0]

In [ ]:
plt.figure(figsize=(10,6))

sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    cmap='coolwarm'
)

plt.show()

In [ ]:
X = df[['distance_km',
        'passenger_count',
        'hour',
        'day',
        'month']]

y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
lr = LinearRegression()

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

lr_r2 = r2_score(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_mae = mean_absolute_error(y_test, y_pred_lr)

print("Linear Regression")
print("R2 Score :", lr_r2)
print("RMSE :", lr_rmse)
print("MAE :", lr_mae)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

results = pd.DataFrame({
    "Model":["Linear Regression","Random Forest"],
    "R2":[
        lr_r2,
        r2_score(y_test, y_pred_rf)
    ],
    "RMSE":[
        lr_rmse,
        np.sqrt(mean_squared_error(y_test, y_pred_rf))
    ],
    "MAE":[
        lr_mae,
        mean_absolute_error(y_test, y_pred_rf)
    ]
})

results